# PlotPal AI — Fine-tuning Qwen2.5-VL-3B for Vacant Space Detection

**Goal:** Fine-tune a 3B parameter Vision Language Model to excel at identifying vacant/underutilized urban spaces from satellite imagery and recommending suitable infrastructure types.

**Model:** `Qwen/Qwen2.5-VL-3B-Instruct` (3.1B params, dynamic resolution)

**Framework:** Unsloth (2x faster, 60% less VRAM)

**Datasets:**
- Remote Sensing VQA (17K samples with "bare land" class)
- RSICD Captions (10K+ satellite image-caption pairs)
- Our Mumbai pipeline results (500 cells → converted to instruction format)
- Synthetically augmented samples from NWPU-RESISC45

**Hardware:** Google Colab T4 (16GB) with QLoRA 4-bit

---

## Cell 1: Install Dependencies

In [ ]:
%%capture
!pip install unsloth
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git
!pip install datasets pillow requests tqdm

## Cell 2: Check GPU & VRAM

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")

## Cell 3: Load Model with Unsloth (4-bit QLoRA)

In [ ]:
from unsloth import FastVisionModel

model, tokenizer = FastVisionModel.from_pretrained(
    "Qwen/Qwen2.5-VL-3B-Instruct",
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,     # Fine-tune vision encoder too
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,                            # LoRA rank
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

print(f"\nTrainable parameters: {model.print_trainable_parameters()}")

---
## Cell 4: Define Dataset Preparation Functions

We need to convert multiple data sources into Qwen2.5-VL's conversation format:
```
{"role": "user", "content": [{"type": "image", "image": <PIL>}, {"type": "text", "text": "..."}]}
{"role": "assistant", "content": [{"type": "text", "text": "..."}]}
```

In [ ]:
import json
import random
from PIL import Image
from io import BytesIO
import requests
from pathlib import Path

BUILDING_TYPES = [
    "cafe", "mall", "park", "residential", "office",
    "hospital", "school", "gym", "restaurant", "hotel", "retail"
]

# ── System prompt for our task ──
SYSTEM_PROMPT = """You are an expert urban planner analyzing satellite imagery to identify vacant or underutilized spaces suitable for infrastructure development."""

# ── Prompt templates for variety ──
ANALYSIS_PROMPTS = [
    "Analyze this satellite image and identify any vacant or underutilized spaces. For each space found, recommend which infrastructure types from [cafe, mall, park, residential, office, hospital, school, gym, restaurant, hotel, retail] would be most suitable. Return your analysis as JSON.",
    "Look at this satellite/aerial image carefully. Find 2-4 vacant, abandoned, or underutilized areas. For each, suggest what type of building or facility would best serve the community. Respond in JSON format.",
    "As an urban planning expert, examine this satellite view. Identify empty lots, abandoned structures, or underused land. Recommend suitable infrastructure from: cafe, mall, park, residential, office, hospital, school, gym, restaurant, hotel, retail. Output JSON only.",
    "Study this aerial/satellite image of an urban area. Detect vacant spaces and recommend appropriate development types based on surrounding context, lot size, and neighborhood needs. Return structured JSON.",
    "Analyze this satellite image for potential development sites. Identify underutilized spaces and suggest which of these infrastructure types would be most appropriate: cafe, mall, park, residential, office, hospital, school, gym, restaurant, hotel, retail. Provide JSON output.",
]

# ── Scene classification prompts ──
SCENE_PROMPTS = [
    "What type of area does this satellite image show? Describe the land use and any notable features.",
    "Classify this satellite image. Is this area primarily urban, suburban, industrial, agricultural, or vacant? Describe what you see.",
    "Describe this aerial view. What land cover types are visible? Is there any vacant or undeveloped land?",
]

# ── VQA prompts ──
VQA_PROMPTS = [
    "Is there any vacant or undeveloped land visible in this satellite image?",
    "What percentage of this area appears to be built-up vs. open/vacant?",
    "Are there any abandoned or underutilized structures visible?",
    "What type of infrastructure would benefit this area the most?",
    "Describe the road network and accessibility of any open spaces visible.",
]

print("Dataset preparation functions defined.")

## Cell 5: Dataset Source 1 — Remote Sensing VQA Dataset

In [ ]:
from datasets import load_dataset

# Load Remote Sensing VQA dataset
print("Loading Remote Sensing VQA dataset...")
rs_vqa = load_dataset("WaltonFuture/remote-sensing-VQA", split="train")
print(f"Total samples: {len(rs_vqa)}")
print(f"Columns: {rs_vqa.column_names}")
print(f"\nSample: {rs_vqa[0]}")

In [ ]:
def convert_rs_vqa(sample):
    """Convert RS-VQA sample to Qwen2.5-VL conversation format."""
    image = sample["image"]
    question = sample.get("question", sample.get("text", ""))
    answer = sample.get("answer", sample.get("label", ""))

    if isinstance(image, str):
        # If image is a URL or path
        try:
            image = Image.open(requests.get(image, stream=True).raw) if image.startswith("http") else Image.open(image)
        except:
            return None

    if image is None or not isinstance(image, Image.Image):
        return None

    image = image.convert("RGB")

    return {
        "messages": [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
            {"role": "user", "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": question}
            ]},
            {"role": "assistant", "content": [{"type": "text", "text": str(answer)}]}
        ]
    }

# Convert dataset
rs_vqa_converted = []
for i, sample in enumerate(rs_vqa):
    converted = convert_rs_vqa(sample)
    if converted:
        rs_vqa_converted.append(converted)
    if (i + 1) % 2000 == 0:
        print(f"  Converted {i+1}/{len(rs_vqa)}...")

print(f"\nRS-VQA: {len(rs_vqa_converted)} samples converted")

## Cell 6: Dataset Source 2 — RSICD Captions (Scene Understanding)

In [ ]:
# Load RSICD captions dataset for scene understanding
print("Loading RSICD dataset...")
try:
    rsicd = load_dataset("arampacha/rsicd", split="train")
    print(f"RSICD samples: {len(rsicd)}")
    print(f"Columns: {rsicd.column_names}")
except Exception as e:
    print(f"RSICD not available directly, trying alternative...")
    rsicd = None

# Also load UCMerced captions for more satellite scene descriptions
try:
    ucm = load_dataset("jonathan-roberts1/UCMerced-Land-Use", split="train")
    print(f"UCMerced samples: {len(ucm)}")
    print(f"Columns: {ucm.column_names}")
except Exception as e:
    print(f"UCMerced: {e}")
    ucm = None

In [ ]:
def convert_scene_caption(image, caption, label=None):
    """Convert a scene classification/caption sample to conversation format."""
    if image is None or not isinstance(image, Image.Image):
        return None

    image = image.convert("RGB")
    prompt = random.choice(SCENE_PROMPTS)

    # Enrich the answer with vacant land relevance
    answer = caption if caption else f"This is a {label} area."

    # Add vacant land analysis for relevant classes
    vacant_classes = ["bare land", "barren", "desert", "meadow", "sparse residential",
                      "parking lot", "industrial", "storage tanks", "construction"]
    if label and any(vc in str(label).lower() for vc in vacant_classes):
        types = random.sample(BUILDING_TYPES, random.randint(2, 4))
        answer += f" This area contains potential vacant or underutilized space that could be suitable for development such as: {', '.join(types)}."

    return {
        "messages": [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
            {"role": "user", "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt}
            ]},
            {"role": "assistant", "content": [{"type": "text", "text": answer}]}
        ]
    }

scene_converted = []

# Convert RSICD
if rsicd:
    for i, sample in enumerate(rsicd):
        captions = sample.get("captions", [sample.get("text", "")])
        caption = captions[0] if isinstance(captions, list) and captions else str(captions)
        converted = convert_scene_caption(
            sample.get("image"), caption, sample.get("label", None)
        )
        if converted:
            scene_converted.append(converted)
        if (i+1) % 2000 == 0:
            print(f"  RSICD: {i+1}/{len(rsicd)}")

# Convert UCMerced
if ucm:
    for i, sample in enumerate(ucm):
        label = sample.get("label", "unknown")
        # Map numeric labels to names if needed
        if isinstance(label, int) and hasattr(ucm.features.get("label", None), "names"):
            label = ucm.features["label"].names[label]
        converted = convert_scene_caption(
            sample.get("image"), f"This satellite image shows a {label} area.", str(label)
        )
        if converted:
            scene_converted.append(converted)

print(f"Scene/Caption data: {len(scene_converted)} samples converted")

## Cell 7: Dataset Source 3 — Mumbai Pipeline Results (Our Data)

Upload `output/results/` and `output/images/` from the research pipeline to Google Drive or Colab files.

In [ ]:
# Mount Google Drive (upload your output/ folder there first)
from google.colab import drive
drive.mount('/content/drive')

# Set paths — adjust these to match your Drive structure
DRIVE_BASE = "/content/drive/MyDrive/plotpal-research"
RESULTS_DIR = f"{DRIVE_BASE}/results"
IMAGES_DIR = f"{DRIVE_BASE}/images"

import os
if os.path.exists(RESULTS_DIR):
    result_files = [f for f in os.listdir(RESULTS_DIR) if f.endswith('.json')]
    print(f"Found {len(result_files)} result files")
    image_files = [f for f in os.listdir(IMAGES_DIR) if f.endswith('.png')] if os.path.exists(IMAGES_DIR) else []
    print(f"Found {len(image_files)} image files")
else:
    print(f"⚠️ Results directory not found at {RESULTS_DIR}")
    print("Please upload your output/results/ and output/images/ to Google Drive")
    print("Or adjust DRIVE_BASE path above")
    result_files = []
    image_files = []

In [ ]:
def convert_mumbai_result(result_path, images_dir):
    """Convert a Mumbai pipeline result JSON to training conversations."""
    with open(result_path, 'r') as f:
        result = json.load(f)

    cell_id = result["cellId"]
    image_path = os.path.join(images_dir, f"{cell_id}.png")

    if not os.path.exists(image_path):
        return []

    try:
        image = Image.open(image_path).convert("RGB")
    except:
        return []

    conversations = []
    filtered = result.get("filteredResult", result.get("pipelineResult", {}))

    # ── Conversation 1: Full analysis (primary training signal) ──
    prompt = random.choice(ANALYSIS_PROMPTS)
    center = result["center"]
    prompt_with_context = f"Cell Center: {center['lat']:.6f}, {center['lng']:.6f}\nCell Size: 500m × 500m\n\n{prompt}"

    answer_json = json.dumps(filtered, indent=2)

    conversations.append({
        "messages": [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
            {"role": "user", "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt_with_context}
            ]},
            {"role": "assistant", "content": [{"type": "text", "text": answer_json}]}
        ]
    })

    # ── Conversation 2: VQA about the specific spaces ──
    spaces = filtered.get("vacantSpaces", [])
    if spaces:
        num_spaces = len(spaces)
        all_types = []
        for s in spaces:
            all_types.extend(s.get("recommendedTypes", []))
        unique_types = list(set(all_types))

        vqa_q = "How many vacant or underutilized spaces can you identify in this satellite image? What types of infrastructure would you recommend?"
        vqa_a = f"I can identify {num_spaces} vacant or underutilized spaces in this image. "
        vqa_a += f"Based on the surrounding context, the recommended infrastructure types are: {', '.join(unique_types)}. "
        vqa_a += filtered.get("analysis", "")

        conversations.append({
            "messages": [
                {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
                {"role": "user", "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": vqa_q}
                ]},
                {"role": "assistant", "content": [{"type": "text", "text": vqa_a}]}
            ]
        })

    # ── Conversation 3: Per-space detailed analysis ──
    for space in spaces[:2]:  # Limit to 2 per cell to avoid data imbalance
        detail_q = f"Describe the vacant space located at approximately {space['coordinates']['lat']:.4f}, {space['coordinates']['lng']:.4f}. What makes it suitable for development?"
        detail_a = f"Location: {space['location']}\n\n{space['description']}\n\n"
        detail_a += f"Suitability: {space['suitability']}/100\n"
        detail_a += f"Recommended types: {', '.join(space.get('recommendedTypes', []))}\n"
        detail_a += f"Reasons: {'; '.join(space.get('reasons', []))}\n"
        detail_a += f"Considerations: {'; '.join(space.get('considerations', []))}"

        conversations.append({
            "messages": [
                {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
                {"role": "user", "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": detail_q}
                ]},
                {"role": "assistant", "content": [{"type": "text", "text": detail_a}]}
            ]
        })

    return conversations


# Convert all Mumbai results
mumbai_converted = []
if result_files:
    for i, fname in enumerate(result_files):
        convs = convert_mumbai_result(
            os.path.join(RESULTS_DIR, fname), IMAGES_DIR
        )
        mumbai_converted.extend(convs)
        if (i+1) % 100 == 0:
            print(f"  Mumbai: {i+1}/{len(result_files)} files → {len(mumbai_converted)} conversations")

print(f"\nMumbai pipeline: {len(mumbai_converted)} training conversations")
print(f"  From {len(result_files)} cell results")

## Cell 8: Dataset Source 4 — Existing Remote Sensing Instruction Dataset

In [ ]:
# Load AdaptLLM's remote sensing visual instructions
print("Loading remote sensing instruction dataset...")
try:
    rs_instruct = load_dataset(
        "AdaptLLM/remote-sensing-visual-instructions",
        split="train",
        streaming=True  # Stream to avoid memory issues
    )

    # Take a subset (5000 samples) to keep training manageable
    rs_instruct_samples = []
    for i, sample in enumerate(rs_instruct):
        if i >= 5000:
            break
        rs_instruct_samples.append(sample)
        if (i+1) % 1000 == 0:
            print(f"  Loaded {i+1}/5000...")

    print(f"RS Instruction samples loaded: {len(rs_instruct_samples)}")
    if rs_instruct_samples:
        print(f"Columns: {list(rs_instruct_samples[0].keys())}")
except Exception as e:
    print(f"Could not load RS instruction dataset: {e}")
    rs_instruct_samples = []

In [ ]:
def convert_rs_instruct(sample):
    """Convert AdaptLLM RS instruction sample to conversation format."""
    image = sample.get("image")
    if image is None:
        return None

    if isinstance(image, str):
        try:
            if image.startswith("http"):
                image = Image.open(requests.get(image, stream=True, timeout=10).raw)
            else:
                image = Image.open(image)
        except:
            return None

    if not isinstance(image, Image.Image):
        return None

    image = image.convert("RGB")

    # Handle different conversation formats
    conversations = sample.get("conversations", [])
    if not conversations:
        q = sample.get("question", sample.get("text", ""))
        a = sample.get("answer", sample.get("label", ""))
        if not q or not a:
            return None
        conversations = [
            {"from": "human", "value": q},
            {"from": "gpt", "value": a}
        ]

    # Convert to Qwen format
    messages = [{"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]}]

    for j, turn in enumerate(conversations):
        role_from = turn.get("from", turn.get("role", ""))
        value = turn.get("value", turn.get("content", ""))

        # Clean up image tokens from text
        value = value.replace("<image>\n", "").replace("<image>", "").strip()

        if role_from in ("human", "user"):
            content = []
            if j == 0:  # First user turn gets the image
                content.append({"type": "image", "image": image})
            content.append({"type": "text", "text": value})
            messages.append({"role": "user", "content": content})
        elif role_from in ("gpt", "assistant"):
            messages.append({"role": "assistant", "content": [{"type": "text", "text": value}]})

    if len(messages) < 3:  # Need at least system + user + assistant
        return None

    return {"messages": messages}

rs_instruct_converted = []
for i, sample in enumerate(rs_instruct_samples):
    converted = convert_rs_instruct(sample)
    if converted:
        rs_instruct_converted.append(converted)
    if (i+1) % 1000 == 0:
        print(f"  Converted {i+1}/{len(rs_instruct_samples)}...")

print(f"RS Instruction: {len(rs_instruct_converted)} samples converted")

## Cell 9: Combine & Shuffle All Datasets

In [ ]:
# Combine all data sources
all_training_data = []

# Add each source with weights
print("Combining datasets:")
print(f"  RS-VQA:           {len(rs_vqa_converted)} samples")
all_training_data.extend(rs_vqa_converted)

print(f"  Scene/Caption:     {len(scene_converted)} samples")
all_training_data.extend(scene_converted)

print(f"  Mumbai Pipeline:   {len(mumbai_converted)} samples (3x weight)")
# Upsample Mumbai data 3x — it's our primary domain
all_training_data.extend(mumbai_converted * 3)

print(f"  RS Instructions:   {len(rs_instruct_converted)} samples")
all_training_data.extend(rs_instruct_converted)

# Shuffle
random.seed(42)
random.shuffle(all_training_data)

print(f"\n{'='*50}")
print(f"TOTAL TRAINING SAMPLES: {len(all_training_data)}")
print(f"{'='*50}")

# Train/val split (95/5)
split_idx = int(len(all_training_data) * 0.95)
train_data = all_training_data[:split_idx]
val_data = all_training_data[split_idx:]
print(f"Train: {len(train_data)} | Val: {len(val_data)}")

## Cell 10: Create HuggingFace Dataset from Conversations

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

print(f"Train dataset: {train_dataset}")
print(f"Val dataset: {val_dataset}")
print(f"\nSample conversation structure:")
sample = train_dataset[0]
for msg in sample["messages"]:
    role = msg["role"]
    content_types = [c["type"] for c in msg["content"]]
    text_preview = next((c["text"][:80] for c in msg["content"] if c["type"] == "text"), "")
    print(f"  {role}: [{', '.join(content_types)}] {text_preview}...")

## Cell 11: Configure Training with Unsloth

In [ ]:
from unsloth import is_bfloat16_supported
from trl import SFTTrainer, SFTConfig

# Training configuration optimized for T4 16GB
sft_config = SFTConfig(
    # Output
    output_dir="plotpal-vlm-finetune",

    # Batch size — keep small for T4
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,  # Effective batch size = 8

    # Learning rate
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,

    # Training duration
    num_train_epochs=2,
    # max_steps=500,  # Uncomment to limit steps for testing

    # Precision
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),

    # Memory optimization
    gradient_checkpointing=True,
    optim="adamw_8bit",
    max_grad_norm=1.0,

    # Sequence length
    max_seq_length=2048,

    # Logging
    logging_steps=10,
    save_steps=200,
    save_total_limit=3,
    eval_strategy="steps",
    eval_steps=200,

    # Dataset
    dataset_text_field="",
    dataset_num_proc=2,
    remove_unused_columns=False,

    # Seed
    seed=42,

    # Report
    report_to="none",
)

print("Training config ready.")
print(f"  Effective batch size: {sft_config.per_device_train_batch_size * sft_config.gradient_accumulation_steps}")
print(f"  Learning rate: {sft_config.learning_rate}")
print(f"  Epochs: {sft_config.num_train_epochs}")
print(f"  Max seq length: {sft_config.max_seq_length}")

## Cell 12: Initialize Trainer & Start Training

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

print(f"Trainer initialized.")
print(f"Total training samples: {len(train_dataset)}")
print(f"Steps per epoch: {len(train_dataset) // (sft_config.per_device_train_batch_size * sft_config.gradient_accumulation_steps)}")

In [ ]:
# ── VRAM check before training ──
gpu_stats = torch.cuda.get_device_properties(0)
reserved = torch.cuda.memory_reserved(0) / 1e9
allocated = torch.cuda.memory_allocated(0) / 1e9
print(f"GPU: {gpu_stats.name}")
print(f"VRAM Total: {gpu_stats.total_mem / 1e9:.1f} GB")
print(f"VRAM Reserved: {reserved:.1f} GB")
print(f"VRAM Allocated: {allocated:.1f} GB")
print(f"VRAM Free: {gpu_stats.total_mem / 1e9 - reserved:.1f} GB")
print("\nStarting training...")

In [ ]:
# 🚀 TRAIN!
train_result = trainer.train()

# Print results
print("\n" + "="*50)
print("TRAINING COMPLETE")
print("="*50)
print(f"Training loss: {train_result.training_loss:.4f}")
print(f"Total steps: {train_result.global_step}")
print(f"Training time: {train_result.metrics.get('train_runtime', 0)/60:.1f} minutes")

## Cell 13: Save Fine-tuned Model

In [ ]:
# Save LoRA adapter
LOCAL_SAVE_PATH = "plotpal-qwen25vl-3b-lora"
model.save_pretrained(LOCAL_SAVE_PATH)
tokenizer.save_pretrained(LOCAL_SAVE_PATH)
print(f"LoRA adapter saved to {LOCAL_SAVE_PATH}")

# Also save to Google Drive for persistence
DRIVE_SAVE_PATH = "/content/drive/MyDrive/plotpal-research/model"
os.makedirs(DRIVE_SAVE_PATH, exist_ok=True)
model.save_pretrained(DRIVE_SAVE_PATH)
tokenizer.save_pretrained(DRIVE_SAVE_PATH)
print(f"LoRA adapter backed up to {DRIVE_SAVE_PATH}")

In [ ]:
# Optional: Push to Hugging Face Hub
# Uncomment and set your HF token + repo name

# from huggingface_hub import login
# login(token="hf_YOUR_TOKEN")
#
# HF_REPO = "your-username/plotpal-qwen25vl-3b-vacant-space"
# model.push_to_hub(HF_REPO)
# tokenizer.push_to_hub(HF_REPO)
# print(f"Pushed to https://huggingface.co/{HF_REPO}")

---
## Cell 14: Test the Fine-tuned Model

In [ ]:
# Enable inference mode
FastVisionModel.for_inference(model)

def analyze_satellite_image(image_input, prompt=None):
    """Run inference on a satellite image."""
    if isinstance(image_input, str):
        if image_input.startswith("http"):
            image = Image.open(requests.get(image_input, stream=True).raw)
        else:
            image = Image.open(image_input)
    else:
        image = image_input

    image = image.convert("RGB")

    if prompt is None:
        prompt = ANALYSIS_PROMPTS[0]

    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt}
        ]}
    ]

    input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(
        input_text, return_tensors="pt", padding=True
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=1024,
            temperature=0.3,
            do_sample=True,
        )

    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response

print("Inference function ready.")

In [ ]:
# ── Test on Mumbai satellite imagery ──

# Test with a known Mumbai cell image (adjust path as needed)
test_images = []

if os.path.exists(IMAGES_DIR):
    test_files = sorted(os.listdir(IMAGES_DIR))[:3]
    for f in test_files:
        test_images.append(os.path.join(IMAGES_DIR, f))

# Test with an Esri satellite tile URL (Mumbai area)
test_url = "https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/17/75592/74568"
test_images.append(test_url)

print(f"Testing on {len(test_images)} images...\n")

for i, img_path in enumerate(test_images):
    print(f"\n{'='*60}")
    print(f"Test Image {i+1}: {img_path if len(str(img_path)) < 60 else '...' + str(img_path)[-50:]}")
    print("="*60)

    try:
        response = analyze_satellite_image(img_path)
        print(response[:1000])
    except Exception as e:
        print(f"Error: {e}")

## Cell 15: Evaluate Fine-tuned vs. Base Model

In [ ]:
import re

def evaluate_response(response_text):
    """Score a model response on key quality dimensions."""
    scores = {}

    # 1. JSON validity
    json_match = re.search(r'\{[\s\S]*\}', response_text)
    try:
        if json_match:
            parsed = json.loads(json_match.group())
            scores['json_valid'] = 1
            scores['has_vacant_spaces'] = 1 if 'vacantSpaces' in parsed else 0
            spaces = parsed.get('vacantSpaces', [])
            scores['num_spaces'] = len(spaces)

            # Check fields
            if spaces:
                s = spaces[0]
                scores['has_coordinates'] = 1 if 'coordinates' in s else 0
                scores['has_recommended_types'] = 1 if 'recommendedTypes' in s else 0
                scores['has_reasons'] = 1 if 'reasons' in s and len(s.get('reasons', [])) > 0 else 0
                scores['has_description'] = 1 if 'description' in s and len(s.get('description', '')) > 20 else 0

                # Check if recommended types are valid
                valid_types = set(BUILDING_TYPES)
                rec_types = s.get('recommendedTypes', [])
                scores['valid_types'] = 1 if all(t in valid_types for t in rec_types) else 0
        else:
            scores['json_valid'] = 0
    except json.JSONDecodeError:
        scores['json_valid'] = 0

    # 2. Response length (proxy for completeness)
    scores['response_length'] = len(response_text)

    return scores


# Run evaluation on test set
print("Evaluating fine-tuned model on validation samples...\n")
eval_results = []
n_eval = min(20, len(val_data))

for i in range(n_eval):
    sample = val_data[i]
    # Find image in the messages
    image = None
    prompt = None
    for msg in sample["messages"]:
        if msg["role"] == "user":
            for content in msg["content"]:
                if content["type"] == "image":
                    image = content["image"]
                elif content["type"] == "text":
                    prompt = content["text"]

    if image and prompt:
        try:
            response = analyze_satellite_image(image, prompt)
            scores = evaluate_response(response)
            eval_results.append(scores)
            print(f"  [{i+1}/{n_eval}] JSON:{scores.get('json_valid',0)} Spaces:{scores.get('num_spaces',0)} Types:{scores.get('has_recommended_types',0)}")
        except Exception as e:
            print(f"  [{i+1}/{n_eval}] Error: {e}")

# Summary
if eval_results:
    print(f"\n{'='*50}")
    print("EVALUATION SUMMARY")
    print(f"{'='*50}")
    print(f"Samples evaluated: {len(eval_results)}")
    print(f"JSON valid:        {sum(r.get('json_valid',0) for r in eval_results)}/{len(eval_results)} ({100*sum(r.get('json_valid',0) for r in eval_results)/len(eval_results):.0f}%)")
    print(f"Has vacantSpaces:  {sum(r.get('has_vacant_spaces',0) for r in eval_results)}/{len(eval_results)}")
    print(f"Has coordinates:   {sum(r.get('has_coordinates',0) for r in eval_results)}/{len(eval_results)}")
    print(f"Has rec. types:    {sum(r.get('has_recommended_types',0) for r in eval_results)}/{len(eval_results)}")
    print(f"Valid types:       {sum(r.get('valid_types',0) for r in eval_results)}/{len(eval_results)}")
    print(f"Has reasons:       {sum(r.get('has_reasons',0) for r in eval_results)}/{len(eval_results)}")
    print(f"Avg spaces/image:  {sum(r.get('num_spaces',0) for r in eval_results)/len(eval_results):.1f}")
    print(f"Avg response len:  {sum(r.get('response_length',0) for r in eval_results)/len(eval_results):.0f} chars")

## Cell 16: Export Model for Production Use

In [ ]:
# Save as merged 16-bit model (for deployment)
# This merges LoRA weights back into the base model

MERGED_PATH = "/content/drive/MyDrive/plotpal-research/model-merged"

# Option 1: Save merged model in float16
model.save_pretrained_merged(
    MERGED_PATH,
    tokenizer,
    save_method="merged_16bit",
)
print(f"Merged model saved to {MERGED_PATH}")

# Option 2: Save as GGUF for llama.cpp / Ollama deployment
# Uncomment if you want GGUF format:
# model.save_pretrained_gguf(
#     "plotpal-vlm-Q4_K_M",
#     tokenizer,
#     quantization_method="q4_k_m",
# )

print("\nDone! Model ready for deployment.")

---
## Cell 17: Compare Fine-tuned Model vs. Gemini API (Research Comparison)

This cell runs the same test images through both the fine-tuned local model and Gemini API to compare quality.

In [ ]:
import base64

def compare_models(image_path, gemini_api_key=None):
    """Compare fine-tuned model vs Gemini API on the same image."""

    prompt = ANALYSIS_PROMPTS[0]

    # ── Fine-tuned model ──
    print("--- Fine-tuned Qwen2.5-VL-3B ---")
    ft_response = analyze_satellite_image(image_path, prompt)
    ft_scores = evaluate_response(ft_response)
    print(f"JSON valid: {ft_scores.get('json_valid')}, Spaces: {ft_scores.get('num_spaces')}, Types: {ft_scores.get('has_recommended_types')}")
    print(ft_response[:500])

    # ── Gemini API (optional) ──
    if gemini_api_key:
        print("\n--- Gemini 2.5 Flash API ---")
        try:
            from google import genai
            client = genai.Client(api_key=gemini_api_key)

            img = Image.open(image_path) if isinstance(image_path, str) and not image_path.startswith('http') else Image.open(requests.get(image_path, stream=True).raw)
            buf = BytesIO()
            img.save(buf, format='PNG')
            img_b64 = base64.b64encode(buf.getvalue()).decode()

            response = client.models.generate_content(
                model='gemini-2.5-flash',
                contents=[prompt, {"inline_data": {"mime_type": "image/png", "data": img_b64}}]
            )
            gemini_text = response.text
            gemini_scores = evaluate_response(gemini_text)
            print(f"JSON valid: {gemini_scores.get('json_valid')}, Spaces: {gemini_scores.get('num_spaces')}, Types: {gemini_scores.get('has_recommended_types')}")
            print(gemini_text[:500])
        except Exception as e:
            print(f"Gemini error: {e}")

# Run comparison
# Set your Gemini API key here (optional)
GEMINI_KEY = None  # "AIza..."

if test_images:
    compare_models(test_images[0], GEMINI_KEY)

---
## Summary

### What was trained:
- **Base model:** Qwen2.5-VL-3B-Instruct (3.1B params)
- **Method:** QLoRA 4-bit, LoRA rank 16, Unsloth
- **Data:** Multi-source (RS-VQA + RSICD + Mumbai pipeline + RS instructions)
- **Hardware:** Google Colab T4/A100

### Key advantages over Gemini API:
- **Free inference** — no API costs per image
- **Private** — satellite imagery stays local
- **Fast** — direct GPU inference, no network latency
- **Customized** — trained specifically on Mumbai satellite analysis
- **Reproducible** — deterministic outputs for research

### For the research paper:
- Compare fine-tuned 3B model vs. Gemini-2.5-Flash (175B+) API
- Report: JSON validity rate, spatial accuracy, type recommendation accuracy
- Demonstrate that domain-specific fine-tuning of small VLMs can approach large API models
- This validates the pipeline as a complete research contribution

### Files saved:
- `plotpal-qwen25vl-3b-lora/` — LoRA adapter weights
- Google Drive backup at `plotpal-research/model/`
- Optional: merged 16-bit model for deployment